# درس ۰۸ - الگوی طراحی چندعامل (Multi-Agent)


## راه‌اندازی


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## چرا سیستم‌های چندعاملی؟

کارهای دنیای واقعی مانند برنامه‌ریزی سفر شامل انواع مختلفی از تخصص‌ها می‌شود — لجستیک، دانش محلی، بودجه‌بندی و موارد دیگر. یک عامل تنها که سعی می‌کند همه چیز را مدیریت کند، به سرعت غیرقابل کنترل می‌شود.

سیستم‌های چندعاملی این مشکل را از طریق **تخصصی شدن** حل می‌کنند: هر عامل روی یک حوزه تخصصی تمرکز می‌کند و نتایجی با کیفیت بالاتر نسبت به یک فرد عامی ارائه می‌دهد. همچنین آنها **مقیاس‌پذیری** را بهبود می‌بخشند — می‌توانید عامل‌های جدیدی اضافه کنید (مثلاً یک متخصص پرواز، یک منتقد رستوران) بدون اینکه جریان کاری موجود را بازنویسی کنید. عامل‌ها از طریق یک خط لوله ساختاریافته با هم ترکیب می‌شوند و زمینه را از یکی به دیگری منتقل می‌کنند.


## ایجاد عوامل تخصصی


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ساخت یک جریان کاری ترتیبی

`WorkflowBuilder` به شما امکان می‌دهد تا عوامل را به یک گراف جهت‌دار وصل کنید. در اینجا یک خط لوله ساده دو مرحله‌ای ایجاد می‌کنیم: **TravelPlanner** برنامه سفر را پیش‌نویس می‌کند، سپس **TravelConcierge** آن را بازبینی و بهبود می‌بخشد.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## افزودن عوامل بیشتر به جریان کار

یکی از بزرگ‌ترین مزایای الگوی چندعاملی، آسان بودن توسعه آن است. در ادامه یک عامل **BudgetReviewer** اضافه می‌کنیم که طرح را با بودجه مسافر می‌سنجد، مواردی را که ممکن است هزینه‌ها را از حد مجاز فراتر ببرند علامت‌گذاری می‌کند و جایگزین‌های صرفه‌جویی در هزینه را پیشنهاد می‌دهد. جریان کار اکنون سه عامل را به ترتیب اجرا می‌کند:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## خلاصه

در این درس آموختید که چگونه:

1. **ایجاد نمایندگان تخصصی** — هر کدام با نقش متمرکز (برنامه‌ریزی، کانسیرج، بررسی بودجه).
2. **متصل کردن نمایندگان به یک جریان کاری متوالی** با استفاده از `WorkflowBuilder` و `add_edge`.
3. **پخش جریان خروجی** از یک خط لوله چند نماینده‌ای، با ردیابی اینکه کدام نماینده در حال صحبت است.
4. **گسترش یک جریان کاری** با افزودن نمایندگان جدید به زنجیره بدون تغییر نمایندگان موجود.

الگوی طراحی چند نماینده‌ای هر نماینده را ساده نگه می‌دارد در حالی که نتایج غنی‌تر و با بررسی دقیق‌تری نسبت به هر نماینده منفرد تولید می‌کند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
